# **System requirements or Set up**

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [2]:
from google.colab import files

# This opens a file picker in your browser.
# Select the 'kaggle.json' file you downloaded from your Kaggle settings.
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nitinkumardhiman",\r\n"key":"27646c035db30e7aeaf229b5f08e8d0b"}\r\n\r\n'}

# **Load the Dataset**

In [3]:
# Clear out the corrupted zip file completely
import os
if os.path.exists('new-plant-diseases-dataset.zip'):
    os.remove('new-plant-diseases-dataset.zip')
    print("Removed corrupted zip file.")

# Setup Kaggle secure folder
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Restart the download (Do not cancel or stop the cell while this runs)
print("Starting fresh download. This is a 2.7 GB file, please wait until it hits 100%...")
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

# Extract it cleanly once fully downloaded
print("Download finished. Extracting files...")
!unzip -q new-plant-diseases-dataset.zip -d Massive_Plant_Data

print("Enterprise dataset successfully downloaded and extracted!")

Starting fresh download. This is a 2.7 GB file, please wait until it hits 100%...
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [02:40<00:00, 18.1MB/s]

Download finished. Extracting files...
Enterprise dataset successfully downloaded and extracted!


# **Initialize the Data Streams**

In [4]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [5]:
# dataset folder path
data_dir = '/content/Massive_Plant_Data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'

# Heavy "In-The-Wild" augmentations for training (forces model to ignore backgrounds)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Clean standardization for validation (no distortions)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the base dataset with training transformations
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)

# Split 80% for training, 20% for testing validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# CRITICAL FIX: Ensure the validation subset uses clean val_transforms instead of train_transforms
val_dataset.dataset.transform = val_transforms

# Create active streams
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaded {len(full_dataset)} images across {len(full_dataset.classes)} plant disease categories.")

Loaded 70295 images across 38 plant disease categories.


# **Model Architecture**

In [6]:
import torch
import torch.nn as nn
from torchvision import models

In [7]:
# Download a pre-trained ResNet50 model
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze all the early layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer to match our plant disease categories
num_features = model.fc.in_features
num_classes = len(full_dataset.classes)
model.fc = nn.Linear(num_features, num_classes)

# Move the entire model over to the Tesla T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model configured. Custom output layer mapped to {num_classes} distinct disease labels.")
print(f"Model successfully loaded onto device: {device}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s]


Model configured. Custom output layer mapped to 38 distinct disease labels.
Model successfully loaded onto device: cuda


# **The Training Engine**

In [8]:
import time
import copy
import torch.optim as optim
import torch.nn as nn

In [9]:
# Define the Rules of Learning
criterion = nn.CrossEntropyLoss()

# The Optimizer (Adam) updates the weights based on the loss.
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# long training path with a dynamic safety net
num_epochs = 100
patience = 20  # Stop if validation accuracy fails to improve after 20 consecutive epochs

def train_model(model, criterion, optimizer, num_epochs, patience):
    start_time = time.time()

    # Keep track of the best performing model so we can save it
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    # Early Stopping tracking variables
    epochs_no_improve = 0
    early_stop = False

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode (updates weights)
                dataloader = train_loader
            else:
                model.eval()   # Set model to evaluate mode (freezes weights for testing)
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data batches
            for inputs, labels in dataloader:
                # Move data to the Tesla T4 GPU
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients before calculating new ones
                optimizer.zero_grad()

                # Forward Pass: Let the AI guess
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward Pass + Optimize (ONLY if in training phase)
                    if phase == 'train':
                        loss.backward()  # Calculate how wrong the AI was
                        optimizer.step() # Adjust the weights to fix the error

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Calculate Epoch metrics
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Check performance during the validation phase
            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

                    # Reset early stopping counter on improvement
                    epochs_no_improve = 0

                    # Save the physical file so you don't lose progress if Colab crashes
                    torch.save(model.state_dict(), 'enterprise_plant_model.pth')
                    print(f">>> New Best Model Saved! (Acc: {best_acc:.4f})")
                else:
                    # Increment counter if performance flatlined or dropped
                    epochs_no_improve += 1
                    print(f"--> No improvement. Early stopping counter: {epochs_no_improve}/{patience}")

                    if epochs_no_improve >= patience:
                        print(f"\n Early stopping triggered! Accuracy has not changed for {patience} epochs.")
                        early_stop = True

        print() # Line break between epochs

        # Break out of the outer training loop if early stopping condition is met
        if early_stop:
            break

    time_elapsed = time.time() - start_time
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:.4f}')

    # Load the best weights back into the model before returning it
    model.load_state_dict(best_model_wts)
    return model

# RUN THE ENGINE!
model = train_model(model, criterion, optimizer, num_epochs, patience)

Epoch 1/100
----------
Train Loss: 0.5278 Acc: 0.9016
Val Loss: 0.1830 Acc: 0.9587
>>> New Best Model Saved! (Acc: 0.9587)

Epoch 2/100
----------
Train Loss: 0.1522 Acc: 0.9635
Val Loss: 0.1206 Acc: 0.9689
>>> New Best Model Saved! (Acc: 0.9689)

Epoch 3/100
----------
Train Loss: 0.1005 Acc: 0.9754
Val Loss: 0.0954 Acc: 0.9755
>>> New Best Model Saved! (Acc: 0.9755)

Epoch 4/100
----------
Train Loss: 0.0762 Acc: 0.9811
Val Loss: 0.0841 Acc: 0.9758
>>> New Best Model Saved! (Acc: 0.9758)

Epoch 5/100
----------
Train Loss: 0.0616 Acc: 0.9844
Val Loss: 0.0763 Acc: 0.9777
>>> New Best Model Saved! (Acc: 0.9777)

Epoch 6/100
----------
Train Loss: 0.0522 Acc: 0.9868
Val Loss: 0.0743 Acc: 0.9775
--> No improvement. Early stopping counter: 1/20

Epoch 7/100
----------
Train Loss: 0.0449 Acc: 0.9885
Val Loss: 0.0666 Acc: 0.9797
>>> New Best Model Saved! (Acc: 0.9797)

Epoch 8/100
----------
Train Loss: 0.0400 Acc: 0.9898
Val Loss: 0.0614 Acc: 0.9804
>>> New Best Model Saved! (Acc: 0.9804)


KeyboardInterrupt: 

In [10]:
# Load the best weights saved from your previous run
model.load_state_dict(torch.load('enterprise_plant_model.pth'))
print("Successfully loaded your 98.09% accuracy weights! Ready to resume.")

Successfully loaded your 98.09% accuracy weights! Ready to resume.


In [11]:
from sklearn.metrics import classification_report, f1_score
import numpy as np
import torch

print("Evaluating model to calculate F1-Score and Classification Report...")
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

class_names = full_dataset.classes

macro_f1 = f1_score(y_true, y_pred, average='macro')
weighted_f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nF1-Score (Macro): {macro_f1:.4f}")
print(f"F1-Score (Weighted): {weighted_f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

Evaluating model to calculate F1-Score and Classification Report...

F1-Score (Macro): 0.9804
F1-Score (Weighted): 0.9805

Classification Report:
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.99      0.99      0.99       372
                                 Apple___Black_rot       1.00      0.99      1.00       402
                          Apple___Cedar_apple_rust       0.99      1.00      1.00       359
                                   Apple___healthy       1.00      0.99      0.99       416
                               Blueberry___healthy       0.99      1.00      0.99       391
          Cherry_(including_sour)___Powdery_mildew       1.00      0.99      1.00       355
                 Cherry_(including_sour)___healthy       1.00      0.99      0.99       372
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.97      0.94      0.95       297
                       Co

In [12]:
from google.colab import files
files.download('enterprise_plant_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>